# 23 — Deep agents: four disciplines for long horizons

Between 2025 and 2026, agent tasks stretched from a dozen tool calls to hundreds, and the bottleneck moved from the model to the **context window**. The systems that worked at that horizon — Claude Code, Anthropic's multi-agent research system, and the harnesses LangChain later named *deep agents* — converged on the same four disciplines:

1. **Planning as a persistent artifact.** The plan is a todo list the agent re-reads and updates every step, not something it remembers.
2. **Filesystem as context offload.** Intermediate results are written out of the prompt and read back on demand (Anthropic calls this *structured note-taking*).
3. **Sub-agents with isolated context.** A subtask gets a fresh window and returns only a compact conclusion; the bulk of the thinking never enters the parent's window.
4. **Detailed prompts and skills.** A rich system prompt plus a catalog of skills whose full instructions load only when needed (*progressive disclosure*).

Each pillar is a lab pattern you have already met, sharpened by context economics: pillar 1 is notebook 05's state made agent-legible, pillar 3 is notebook 04's delegation plus isolation, and the loop underneath is still notebook 01's.

`lab_core.deep` models the window as a first-class object so isolation can be *measured*, not asserted: a `Context` is a list of labeled entries with a deterministic token estimate (chars/4 — a stand-in for a real tokenizer, stated here honestly). The *trace* stays complete (the flight recorder); the *Context* is curated (what a model would see).

Primary sources: [LangChain — Deep Agents](https://www.langchain.com/blog/deep-agents), [Anthropic — Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents), [Anthropic — How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system), and the [`deepagents` package](https://github.com/langchain-ai/deepagents).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from lab_core.deep import Context, TodoList, VirtualFS, estimate_tokens

# Pillar 1 - the plan is an artifact: tiny, persistent, re-read every step.
todos = TodoList()
todos.add('survey sources', 'digest doc-a', 'write recommendation')
context = Context(owner='parent')
context.pin('todos', todos.render())
print(todos.render(), '\n')

todos.complete(0, '2 sources found')
context.pin('todos', todos.render())          # re-pinned in place, not appended
print(todos.render())
print('\nplan slot costs', estimate_tokens(todos.render()), 'tokens, however long the run gets')

In [ ]:
# Pillar 2 - offloading: the same document, carried in context vs. moved to a file.
doc = 'finding: deterministic spines cut variance.\n' + 'supporting detail paragraph. ' * 55

fs = VirtualFS()
ctx_flat = Context(owner='flat-agent')
ctx_deep = Context(owner='deep-agent')

ctx_flat.add('observation', doc)              # payload enters the window
ctx_deep.add('fs', fs.write('/notes/doc.md', doc))   # only the receipt enters the window

print('document:', len(doc), 'chars')
print('flat context:', ctx_flat.tokens(), 'tokens | deep context:', ctx_deep.tokens(), 'tokens')
print('receipt was:', ctx_deep.entries[-1].text)
print('read back on demand:', fs.read('/notes/doc.md')[:44], '...')

In [ ]:
# Pillar 3 - sub-agent isolation, proven by membership checks in both directions.
from lab_core.deep import SubAgent

parent = Context(owner='parent')
parent.pin('goal', 'PARENT-ONLY: negotiate the budget quietly, then review the doc')

def digest(task, context, files):
    text = files.read('/notes/doc.md')
    context.add('tool:read_file', text[:120])
    return {'summary': text.splitlines()[0], 'detail': text}

report = SubAgent('reader', 'Digest one document.', digest, fs).run('digest /notes/doc.md')

print('sub-agent window saw:', [e.label for e in report.context.entries])
print('parent secret leaked into sub-agent?', 'PARENT-ONLY' in report.context.render())
print('sub-agent bulk leaked into parent?', 'supporting detail' in parent.render())
print('what crossed back:', repr(report.summary), '+', report.detail_path)
print('windows:', parent.tokens(), 'tokens (parent) vs', report.context.tokens(), 'tokens (sub)')

In [ ]:
# Pillar 4 - skills: a cheap catalog, full instructions only on demand.
from lab_core.deep import Skill, compose_system_prompt

skills = [
    Skill('cite-evidence', 'Cite a source path for every claim.',
          'For every claim in your answer, append the /sources path it came from. ' * 12),
    Skill('tabular-summary', 'Summarize comparisons as a table.',
          'When comparing options, render a markdown table with one row per option. ' * 12),
]
prompt = compose_system_prompt('You review documents and write recommendations.', skills)
print(prompt, '\n')
print('catalog costs', estimate_tokens(prompt), 'tokens;',
      'loading cite-evidence would add', estimate_tokens(skills[0].instructions), 'more')

**Trade-off.** Every pillar buys window headroom with indirection: receipts instead of payloads, summaries instead of transcripts, catalogs instead of instructions. The agent must spend a tool call to get detail back, and files it wrote earlier are untrusted input when read back later — notebook 05's gates still apply. Notebook 24 assembles the four pillars into one running agent.